In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import json

df = pd.read_csv('../data/ufc_clean.csv')
print(f'Rijen: {len(df)}')
print(f'Kolommen: {len(df.columns)}')
df.head()

Rijen: 8520
Kolommen: 99


,RedFighter,BlueFighter,Winner,WeightClass,TitleBout,Method,RedWins,RedLosses,RedWinStreak,RedLoseStreak,...,DifAvgClinchLand,DifAvgGroundLand,DifRecentWins,DifRecentAvgSigStrLand,DifRecentAvgSigStrPct,DifRecentAvgTDLand,DifRecentAvgTDPct,DifRecentAvgSubAtt,DifRecentAvgCtrlSec,DifRecentAvgKD
0,Jack Della Maddalena,Carlos Prates,Blue,NaN,False,KO/TKO Round: 3 Time: 3:17 Time format: 5 Rnd ...,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Beneil Dariush,Quillan Salkilld,Blue,NaN,False,KO/TKO Round: 1 Time: 3:29 Time format: 3 Rnd ...,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Tim Elliott,Steve Erceg,Blue,NaN,False,Decision - Unanimous Round: 3 Time: 5:00 Time ...,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Marwan Rahiki,Ollie Schmid,Red,NaN,False,KO/TKO Round: 1 Time: 2:47 Time format: 3 Rnd ...,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Shamil Gaziev,Brando Pericic,Blue,NaN,False,KO/TKO Round: 2 Time: 3:44 Time format: 3 Rnd ...,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df['target'] = (df['Winner'] == 'Red').astype(int)

exclude = ['RedFighter', 'BlueFighter', 'Winner', 'WeightClass', 'Method', 'target']
feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64', 'int64']]

print(f'Aantal features: {len(feature_cols)}')
print(feature_cols)

Aantal features: 93
['RedWins', 'RedLosses', 'RedWinStreak', 'RedLoseStreak', 'RedLongestWinStreak', 'RedKOWins', 'RedSubWins', 'RedDecWins', 'RedTitleBouts', 'RedAvgSigStrLand', 'RedAvgSigStrAtt', 'RedAvgSigStrPct', 'RedAvgTDLand', 'RedAvgTDPct', 'RedAvgSubAtt', 'RedAvgCtrlSec', 'RedAvgKD', 'RedAvgHeadLand', 'RedAvgBodyLand', 'RedAvgLegLand', 'RedAvgDistanceLand', 'RedAvgClinchLand', 'RedAvgGroundLand', 'RedRecentWins', 'RedRecentAvgSigStrLand', 'RedRecentAvgSigStrPct', 'RedRecentAvgTDLand', 'RedRecentAvgTDPct', 'RedRecentAvgSubAtt', 'RedRecentAvgCtrlSec', 'RedRecentAvgKD', 'BlueWins', 'BlueLosses', 'BlueWinStreak', 'BlueLoseStreak', 'BlueLongestWinStreak', 'BlueKOWins', 'BlueSubWins', 'BlueDecWins', 'BlueTitleBouts', 'BlueAvgSigStrLand', 'BlueAvgSigStrAtt', 'BlueAvgSigStrPct', 'BlueAvgTDLand', 'BlueAvgTDPct', 'BlueAvgSubAtt', 'BlueAvgCtrlSec', 'BlueAvgKD', 'BlueAvgHeadLand', 'BlueAvgBodyLand', 'BlueAvgLegLand', 'BlueAvgDistanceLand', 'BlueAvgClinchLand', 'BlueAvgGroundLand', 'BlueRec

In [5]:
X = df[feature_cols].fillna(0)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {len(X_train)} rijen')
print(f'Test set:     {len(X_test)} rijen')

Training set: 6816 rijen
Test set:     1704 rijen


In [7]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%)')

Accuracy: 0.7453 (74.53%)


In [9]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(20)

DifLosses                  0.040465
DifLongestWinStreak        0.028525
DifWins                    0.026545
DifRecentAvgSigStrPct      0.024133
BlueAvgGroundLand          0.023801
DifAvgSigStrPct            0.022797
RedAvgSigStrPct            0.019196
BlueRecentAvgSigStrLand    0.017955
BlueRecentAvgSigStrPct     0.017191
BlueAvgSigStrPct           0.017085
DifAvgDistanceLand         0.016900
RedAvgSigStrAtt            0.016807
DifRecentWins              0.016246
RedRecentAvgSigStrPct      0.016203
RedAvgCtrlSec              0.016008
DifAvgSigStrAtt            0.015964
DifAvgGroundLand           0.015945
DifAvgHeadLand             0.015642
BlueAvgSigStrAtt           0.015621
RedWins                    0.015545
dtype: float64

In [10]:
joblib.dump(model, '../models/model_v3.pkl')

with open('../models/feature_names_v3.json', 'w') as f:
    json.dump(feature_cols, f)

print('✅ Model opgeslagen → models/model_v3.pkl')
print('✅ Features opgeslagen → models/feature_names_v3.json')

✅ Model opgeslagen → models/model_v3.pkl
✅ Features opgeslagen → models/feature_names_v3.json
